In [ ]:
# Retail Sales Forecasting - Regression Project
# Dataset: stores_sales_forecasting.csv
# Encoding note: the supplied CSV is read with cp1252 because it contains
# characters that are not valid UTF-8.

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Step 0: Import Required Libraries

Import pandas, NumPy, Matplotlib, Seaborn, and Scikit-learn libraries required for data analysis, visualization, regression, and evaluation.
```

In [ ]:
# ============================================================
# 1. DATA LOADING
# ============================================================

FILE_PATH = "stores_sales_forecasting.csv"

try:
    df = pd.read_csv(FILE_PATH, encoding="cp1252")
except UnicodeDecodeError:
    df = pd.read_csv(FILE_PATH, encoding="latin1")

print("Dataset shape:", df.shape)
print(df.head())

## Step 1: Data Loading

Load the retail sales CSV file into a pandas DataFrame.

The supplied file may require `cp1252` or `latin1` encoding because some product/customer text contains non-UTF-8 characters.


In [ ]:
# ============================================================
# 2. DATA UNDERSTANDING
# ============================================================

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDataset information:")
df.info()

print("\nDescriptive statistics:")
print(df.describe(include="all").T)

## Step 2: Data Understanding

Inspect:
- Number of rows and columns
- Column names
- Data types
- First and last records
- Descriptive statistics
- Dataset information
```

In [ ]:
# ============================================================
# 3. DATA CLEANING
# ============================================================

df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], errors="coerce")

numeric_cols = ["Sales", "Quantity", "Discount", "Profit"]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("\nMissing values before cleaning:")
print(df.isnull().sum())

df = df.dropna(subset=["Order Date", "Sales"]).copy()
df = df.drop_duplicates().copy()

# Useful time features
df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month
df["Quarter"] = df["Order Date"].dt.quarter
df["Month_Name"] = df["Order Date"].dt.strftime("%b")
df["Day_of_Week"] = df["Order Date"].dt.day_name()

print("\nShape after cleaning:", df.shape)

## Step 3: Data Cleaning

Perform:
- Date conversion
- Numeric conversion
- Missing-value inspection
- Duplicate removal
- Invalid-date removal
- Creation of Year, Month, Quarter, Month Name, and Day of Week features
```

In [ ]:
# ============================================================
# 4. DESCRIPTIVE ANALYSIS
# ============================================================

print("\nSales statistics:")
print(df["Sales"].describe())

print("\nTotal Sales:", round(df["Sales"].sum(), 2))
print("Total Profit:", round(df["Profit"].sum(), 2))
print("Total Quantity:", int(df["Quantity"].sum()))

print("\nSales by Category:")
print(df.groupby("Category")["Sales"].sum().sort_values(ascending=False))

print("\nSales by Region:")
print(df.groupby("Region")["Sales"].sum().sort_values(ascending=False))

print("\nTop 10 Products by Sales:")
print(
    df.groupby("Product Name")["Sales"]
      .sum()
      .sort_values(ascending=False)
      .head(10)
)

## Step 4: Descriptive Analysis

Descriptive analysis summarizes what happened historically.

Calculate:
- Total sales
- Total profit
- Total quantity
- Sales statistics
- Sales by category
- Sales by region
- Top products by sales
```

In [ ]:
# ============================================================
# 5. DIAGNOSTIC ANALYSIS
# ============================================================

category_summary = (
    df.groupby("Category")
      .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
      .sort_values("Sales", ascending=False)
)

print("\nCategory Sales vs Profit:")
print(category_summary)

region_summary = (
    df.groupby("Region")
      .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
      .sort_values("Sales", ascending=False)
)

print("\nRegion Sales vs Profit:")
print(region_summary)

print("\nAverage Sales by Discount level:")
print(df.groupby("Discount")["Sales"].mean().sort_index())

## Step 5: Diagnostic Analysis

Diagnostic analysis investigates why sales and profit differ across business dimensions.

Compare:
- Sales vs Profit by Category
- Sales vs Profit by Region
- Average Sales at different Discount levels

In [ ]:
# ============================================================
# 6. VISUALIZATION
# ============================================================

plt.figure(figsize=(10, 5))
sns.histplot(df["Sales"], bins=40, kde=True)
plt.title("Distribution of Sales")
plt.xlabel("Sales")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
category_sales = df.groupby("Category")["Sales"].sum().sort_values(ascending=False)
sns.barplot(x=category_sales.index, y=category_sales.values)
plt.title("Total Sales by Category")
plt.xlabel("Category")
plt.ylabel("Sales")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
monthly_sales_plot = df.set_index("Order Date").resample("MS")["Sales"].sum()
monthly_sales_plot.plot(marker="o")
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.grid(True)
plt.tight_layout()
plt.show()

## 6. Visualization

Data visualization is used to understand sales patterns, compare product categories, and analyze sales trends over time.

The following visualizations are created:

### 6.1 Distribution of Sales

A histogram is created to show the distribution of transaction-level sales.

* The **X-axis** represents Sales.
* The **Y-axis** represents Frequency.
* The KDE curve shows the overall shape of the sales distribution.

This visualization helps identify the spread, concentration, and distribution of sales values.

### 6.2 Total Sales by Category

A bar chart is created to compare total sales across different product categories.

The sales values are grouped by `Category` and sorted in descending order.

This visualization helps identify which product category generates the highest total sales.

### 6.3 Monthly Sales Trend

Sales are aggregated by month using the `Order Date` column.

A line chart is used to display the monthly sales trend.

* The **X-axis** represents Month.
* The **Y-axis** represents Sales.
* Each point represents the total sales for a particular month.

This visualization helps identify increases, decreases, and overall patterns in monthly sales performance.


In [ ]:
# ============================================================
# 7. CORRELATION ANALYSIS
# ============================================================

corr_cols = ["Sales", "Quantity", "Discount", "Profit"]
corr = df[corr_cols].corr()

print("\nCorrelation matrix:")
print(corr)

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

# Scatter plot for the strongest practical sales driver in this dataset
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="Quantity", y="Sales", alpha=0.6)
plt.title("Quantity vs Sales")
plt.xlabel("Quantity")
plt.ylabel("Sales")
plt.tight_layout()
plt.show()

## 7. Correlation Analysis

Correlation analysis is used to measure the relationship between numerical variables in the sales dataset.

The analysis focuses on the following variables:

* `Sales`
* `Quantity`
* `Discount`
* `Profit`

### 7.1 Create Correlation Matrix

The correlation matrix is calculated using the `.corr()` function.

```python
corr_cols = ["Sales", "Quantity", "Discount", "Profit"]
corr = df[corr_cols].corr()
```

The correlation coefficient ranges from **-1 to +1**:

* **+1** → Strong positive relationship
* **0** → No linear relationship
* **-1** → Strong negative relationship

The correlation matrix is printed to understand the relationships between Sales, Quantity, Discount, and Profit.

### 7.2 Correlation Heatmap

A heatmap is created to visualize the correlation matrix.

```python
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
```

The heatmap displays the correlation values inside each cell.

This visualization makes it easier to identify:

* Positive relationships
* Negative relationships
* Weak relationships
* Strong relationships between variables

The **Sales** relationships are particularly important because the objective of the project is sales forecasting.

### 7.3 Quantity vs Sales

A scatter plot is created to analyze the relationship between `Quantity` and `Sales`.

```python
sns.scatterplot(data=df, x="Quantity", y="Sales", alpha=0.6)
```

* The **X-axis** represents Quantity.
* The **Y-axis** represents Sales.
* Each point represents an individual transaction.

This visualization helps determine whether higher quantities sold are generally associated with higher sales values.

### 7.4 Interpretation

The correlation analysis helps identify variables that may be useful for predictive modeling.

In this dataset, **Quantity has a positive relationship with Sales**, meaning that transactions with higher quantities generally tend to have higher sales values.

The correlation between **Discount and Profit is negative**, indicating that higher discounts are generally associated with lower profit.

Correlation does not necessarily imply causation, but it provides useful information for selecting and understanding features used in the predictive analysis.


In [ ]:
# ============================================================
# 8. TRANSACTION-LEVEL REGRESSION
# Predict Sales from Quantity, Discount, and selected categories.
# ============================================================

transaction_features = [
    col for col in ["Quantity", "Discount", "Category", "Sub-Category", "Region"]
    if col in df.columns
]

X = df[transaction_features]
y = df["Sales"]

categorical_features = [c for c in transaction_features if df[c].dtype == "object"]
numeric_features = [c for c in transaction_features if c not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

transaction_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression()),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

transaction_model.fit(X_train, y_train)
transaction_pred = transaction_model.predict(X_test)

transaction_mae = mean_absolute_error(y_test, transaction_pred)
transaction_rmse = np.sqrt(mean_squared_error(y_test, transaction_pred))
transaction_r2 = r2_score(y_test, transaction_pred)

print("\nTransaction-level Linear Regression")
print("MAE :", round(transaction_mae, 4))
print("RMSE:", round(transaction_rmse, 4))
print("R2  :", round(transaction_r2, 4))

plt.figure(figsize=(7, 6))
plt.scatter(y_test, transaction_pred, alpha=0.6)
plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.title("Actual vs Predicted Sales - Transaction Regression")
plt.tight_layout()
plt.show()

## 8. Transaction-Level Regression

Transaction-level regression is used to predict the **Sales** value of an individual transaction using factors such as Quantity, Discount, Category, Sub-Category, and Region.

The target variable is:

* `Sales`

The input features are:

* `Quantity`
* `Discount`
* `Category`
* `Sub-Category`
* `Region`

### 8.1 Select Transaction Features

The required features are selected from the dataset.

```python
transaction_features = [
    col for col in ["Quantity", "Discount", "Category", "Sub-Category", "Region"]
    if col in df.columns
]
```

This approach ensures that only columns available in the dataset are selected.

The input features (`X`) and target variable (`y`) are then defined:

```python
X = df[transaction_features]
y = df["Sales"]
```

* **X** → Independent variables used for prediction.
* **y** → Dependent variable, which is Sales.

### 8.2 Identify Categorical and Numerical Features

The selected features are divided into categorical and numerical variables.

```python
categorical_features = [
    c for c in transaction_features
    if df[c].dtype == "object"
]

numeric_features = [
    c for c in transaction_features
    if c not in categorical_features
]
```

Numerical features can be used directly by the regression model, while categorical features need to be converted into numerical form.

### 8.3 Data Preprocessing

A `ColumnTransformer` is used to preprocess the input features.

```python
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)
```

The preprocessing performs the following operations:

* Numerical features are passed through without transformation.
* Categorical features are converted into numerical values using **One-Hot Encoding**.
* `handle_unknown="ignore"` allows the model to handle categories that may appear in the test data but were not present in the training data.

### 8.4 Create Linear Regression Pipeline

A machine learning pipeline is created by combining preprocessing and the Linear Regression model.

```python
transaction_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression()),
    ]
)
```

The pipeline ensures that preprocessing and model training are performed consistently.

### 8.5 Split Data into Training and Testing Sets

The dataset is divided into training and testing sets.

```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
```

The dataset is split as follows:

* **80%** → Training data
* **20%** → Testing data

The training data is used to build the model, while the testing data is used to evaluate its performance.

`random_state=42` ensures that the same split can be reproduced.

### 8.6 Train the Regression Model

The Linear Regression model is trained using the training dataset.

```python
transaction_model.fit(X_train, y_train)
```

The trained model learns the relationship between the selected transaction features and Sales.

### 8.7 Predict Sales

The trained model is used to predict Sales for the test dataset.

```python
transaction_pred = transaction_model.predict(X_test)
```

The predicted values are then compared with the actual Sales values.

### 8.8 Model Evaluation

Three evaluation metrics are calculated:

```python
transaction_mae = mean_absolute_error(y_test, transaction_pred)
transaction_rmse = np.sqrt(mean_squared_error(y_test, transaction_pred))
transaction_r2 = r2_score(y_test, transaction_pred)
```

#### Mean Absolute Error (MAE)

MAE measures the average absolute difference between actual and predicted Sales.

* Lower MAE indicates better prediction accuracy.
* It is expressed in the same units as Sales.

#### Root Mean Squared Error (RMSE)

RMSE measures the square root of the average squared prediction error.

* Lower RMSE indicates better model performance.
* RMSE gives more importance to larger prediction errors.

#### R² Score

R² measures how much of the variation in Sales is explained by the regression model.

* **R² = 1** → Perfect prediction
* **R² = 0** → Model does not explain the variation better than a simple mean-based prediction
* Higher R² generally indicates a better fit.

### 8.9 Display Model Performance

The calculated evaluation metrics are displayed:

```python
print("\nTransaction-level Linear Regression")
print("MAE :", round(transaction_mae, 4))
print("RMSE:", round(transaction_rmse, 4))
print("R2  :", round(transaction_r2, 4))
```

These values are used to understand the accuracy and performance of the transaction-level regression model.

### 8.10 Actual vs Predicted Sales Visualization

A scatter plot is created to compare actual Sales with predicted Sales.

```python
plt.scatter(y_test, transaction_pred, alpha=0.6)
```

* The **X-axis** represents Actual Sales.
* The **Y-axis** represents Predicted Sales.
* Each point represents a transaction from the test dataset.

If the predicted values are close to the actual values, the points will show a clear pattern around an imaginary diagonal line.

This visualization helps identify how closely the regression model's predictions match the actual transaction-level Sales values.

### 8.11 Conclusion

Transaction-level Linear Regression provides a predictive approach for estimating Sales based on transaction characteristics.

The model uses both numerical and categorical features, with categorical variables converted using One-Hot Encoding. The performance is evaluated using **MAE, RMSE, and R²**, while the Actual vs Predicted scatter plot provides a visual assessment of prediction accuracy.


In [ ]:
# ============================================================
# 9. MONTHLY SALES FORECASTING USING LAG REGRESSION
# ============================================================
# This is the main forecasting section.
# Sales are aggregated by month and lagged sales are used as predictors.

monthly = (
    df.set_index("Order Date")
      .resample("MS")["Sales"]
      .sum()
      .reset_index()
      .rename(columns={"Sales": "Monthly_Sales"})
)

monthly["Time_Index"] = np.arange(len(monthly))
monthly["Year"] = monthly["Order Date"].dt.year
monthly["Month"] = monthly["Order Date"].dt.month
monthly["Quarter"] = monthly["Order Date"].dt.quarter
monthly["Lag_1"] = monthly["Monthly_Sales"].shift(1)
monthly["Lag_2"] = monthly["Monthly_Sales"].shift(2)
monthly["Lag_3"] = monthly["Monthly_Sales"].shift(3)
monthly["Rolling_3"] = monthly["Monthly_Sales"].shift(1).rolling(3).mean()

forecast_features = [
    "Time_Index", "Year", "Month", "Quarter",
    "Lag_1", "Lag_2", "Lag_3", "Rolling_3"
]

forecast_data = monthly.dropna().copy()

# Time-order split: do not randomly shuffle time-series observations.
split_index = int(len(forecast_data) * 0.80)
train = forecast_data.iloc[:split_index].copy()
test = forecast_data.iloc[split_index:].copy()

forecast_model = LinearRegression()
forecast_model.fit(train[forecast_features], train["Monthly_Sales"])

test_pred = forecast_model.predict(test[forecast_features])

forecast_mae = mean_absolute_error(test["Monthly_Sales"], test_pred)
forecast_rmse = np.sqrt(mean_squared_error(test["Monthly_Sales"], test_pred))
forecast_r2 = r2_score(test["Monthly_Sales"], test_pred)

print("\nMonthly Sales Forecast Regression")
print("MAE :", round(forecast_mae, 4))
print("RMSE:", round(forecast_rmse, 4))
print("R2  :", round(forecast_r2, 4))

test_results = test[["Order Date", "Monthly_Sales"]].copy()
test_results["Predicted_Sales"] = test_pred

print("\nActual vs Predicted monthly sales:")
print(test_results.to_string(index=False))

plt.figure(figsize=(12, 5))
plt.plot(train["Order Date"], train["Monthly_Sales"], label="Train")
plt.plot(test["Order Date"], test["Monthly_Sales"], label="Actual Test", marker="o")
plt.plot(test["Order Date"], test_pred, label="Predicted Test", marker="x")
plt.title("Monthly Sales - Actual vs Predicted")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 9. Monthly Sales Forecasting Using Lag Regression

Monthly Sales Forecasting is the main predictive forecasting section of this project.

Instead of predicting individual transaction Sales, the transaction data is aggregated into **monthly sales totals**. Previous months' sales are then used as predictors to forecast future monthly sales.

### 9.1 Aggregate Sales by Month

The transaction-level Sales data is grouped by month using the `Order Date` column.

```python
monthly = (
    df.set_index("Order Date")
      .resample("MS")["Sales"]
      .sum()
      .reset_index()
      .rename(columns={"Sales": "Monthly_Sales"})
)
```

The `MS` frequency represents the **start of each month**.

The resulting dataset contains:

* `Order Date` → Month
* `Monthly_Sales` → Total sales for that month

This converts the original transaction-level dataset into a monthly time-series dataset.

### 9.2 Create Time-Based Features

Additional time-related features are created to help the regression model identify seasonal and time-based patterns.

```python
monthly["Time_Index"] = np.arange(len(monthly))
monthly["Year"] = monthly["Order Date"].dt.year
monthly["Month"] = monthly["Order Date"].dt.month
monthly["Quarter"] = monthly["Order Date"].dt.quarter
```

The features represent:

* **Time_Index** → Sequential position of each month.
* **Year** → Calendar year.
* **Month** → Month number from 1 to 12.
* **Quarter** → Quarter number from 1 to 4.

These features help the model capture long-term trends and seasonal patterns.

### 9.3 Create Lag Features

Previous monthly Sales values are used as predictors.

```python
monthly["Lag_1"] = monthly["Monthly_Sales"].shift(1)
monthly["Lag_2"] = monthly["Monthly_Sales"].shift(2)
monthly["Lag_3"] = monthly["Monthly_Sales"].shift(3)
```

The lag features represent:

* **Lag_1** → Sales from the previous month.
* **Lag_2** → Sales from two months earlier.
* **Lag_3** → Sales from three months earlier.

Lag features are useful in time-series forecasting because previous sales can provide information about future sales.

### 9.4 Create Rolling Average Feature

A three-month rolling average is calculated using previous monthly sales.

```python
monthly["Rolling_3"] = (
    monthly["Monthly_Sales"]
    .shift(1)
    .rolling(3)
    .mean()
)
```

The `Rolling_3` feature represents the average Sales of the previous three months.

Using `.shift(1)` ensures that the current month's Sales are not included when calculating its predictor.

This helps reduce the risk of using future information when predicting the current month.

### 9.5 Select Forecasting Features

The features used by the forecasting model are defined as:

```python
forecast_features = [
    "Time_Index", "Year", "Month", "Quarter",
    "Lag_1", "Lag_2", "Lag_3", "Rolling_3"
]
```

The model uses:

* Time trend
* Year
* Month
* Quarter
* Previous month's Sales
* Sales from two months earlier
* Sales from three months earlier
* Three-month rolling average

The target variable is:

```text
Monthly_Sales
```

### 9.6 Remove Missing Values

Lag and rolling calculations create missing values at the beginning of the dataset.

These rows are removed before training the model.

```python
forecast_data = monthly.dropna().copy()
```

This ensures that the forecasting model receives complete feature values.

### 9.7 Split the Time-Series Data

The data is divided into training and testing datasets using chronological order.

```python
split_index = int(len(forecast_data) * 0.80)

train = forecast_data.iloc[:split_index].copy()
test = forecast_data.iloc[split_index:].copy()
```

The dataset is divided into:

* **80%** → Training data
* **20%** → Testing data

Unlike ordinary machine learning problems, the time-series data is **not randomly shuffled**.

The earlier observations are used for training, while later observations are used for testing. This better represents a real forecasting situation where the model uses historical data to predict future data.

### 9.8 Train the Forecasting Model

A Linear Regression model is created and trained using the selected forecasting features.

```python
forecast_model = LinearRegression()

forecast_model.fit(
    train[forecast_features],
    train["Monthly_Sales"]
)
```

The model learns the relationship between historical sales patterns, time features, lag values, and monthly Sales.

### 9.9 Predict Test-Period Sales

The trained model is used to predict Sales for the test period.

```python
test_pred = forecast_model.predict(
    test[forecast_features]
)
```

The predicted monthly Sales values are then compared with the actual monthly Sales values.

### 9.10 Evaluate Forecasting Performance

The forecasting model is evaluated using three metrics:

```python
forecast_mae = mean_absolute_error(
    test["Monthly_Sales"],
    test_pred
)

forecast_rmse = np.sqrt(
    mean_squared_error(
        test["Monthly_Sales"],
        test_pred
    )
)

forecast_r2 = r2_score(
    test["Monthly_Sales"],
    test_pred
)
```

#### Mean Absolute Error (MAE)

MAE measures the average absolute difference between the actual and predicted monthly Sales.

A lower MAE indicates better forecasting accuracy.

#### Root Mean Squared Error (RMSE)

RMSE measures the square root of the average squared prediction error.

A lower RMSE indicates better model performance. RMSE gives greater importance to larger forecasting errors.

#### R² Score

R² measures how much of the variation in monthly Sales is explained by the forecasting model.

A higher R² generally indicates a better fit to the observed data.

### 9.11 Display Forecasting Results

The evaluation metrics are displayed:

```python
print("\nMonthly Sales Forecast Regression")
print("MAE :", round(forecast_mae, 4))
print("RMSE:", round(forecast_rmse, 4))
print("R2  :", round(forecast_r2, 4))
```

These metrics provide a quantitative measure of how accurately the model forecasts monthly Sales.

### 9.12 Compare Actual and Predicted Monthly Sales

A results table is created containing the actual and predicted monthly Sales values.

```python
test_results = test[["Order Date", "Monthly_Sales"]].copy()
test_results["Predicted_Sales"] = test_pred
```

The table contains:

* `Order Date`
* `Monthly_Sales` → Actual monthly Sales
* `Predicted_Sales` → Model forecast

The results are printed to allow a month-by-month comparison between actual and predicted Sales.

### 9.13 Actual vs Predicted Monthly Sales Visualization

A line chart is created to compare the training data,


In [ ]:
# ============================================================
# 10. FUTURE SALES FORECAST
# ============================================================
# Recursive 3-month forecast. Each predicted month becomes a lag
# for the following month.

future_steps = 3
history = monthly[["Order Date", "Monthly_Sales"]].copy()

future_rows = []

for _ in range(future_steps):
    next_date = history["Order Date"].max() + pd.offsets.MonthBegin(1)

    values = history["Monthly_Sales"].tolist()
    lag1 = values[-1]
    lag2 = values[-2]
    lag3 = values[-3]
    rolling3 = np.mean(values[-3:])

    row = pd.DataFrame([{
        "Order Date": next_date,
        "Time_Index": len(history),
        "Year": next_date.year,
        "Month": next_date.month,
        "Quarter": next_date.quarter,
        "Lag_1": lag1,
        "Lag_2": lag2,
        "Lag_3": lag3,
        "Rolling_3": rolling3,
    }])

    predicted_sales = forecast_model.predict(row[forecast_features])[0]
    predicted_sales = max(0, predicted_sales)

    row["Monthly_Sales"] = predicted_sales
    history = pd.concat(
        [history, row[["Order Date", "Monthly_Sales"]]],
        ignore_index=True
    )
    future_rows.append([next_date, predicted_sales])

future_forecast = pd.DataFrame(
    future_rows, columns=["Forecast_Month", "Forecast_Sales"]
)

print("\nNext 3 months sales forecast:")
print(future_forecast.to_string(index=False))

plt.figure(figsize=(12, 5))
plt.plot(monthly["Order Date"], monthly["Monthly_Sales"], label="Historical Sales")
plt.plot(
    future_forecast["Forecast_Month"],
    future_forecast["Forecast_Sales"],
    marker="o",
    linestyle="--",
    label="Forecast"
)
plt.title("Future Monthly Sales Forecast")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 10. Future Sales Forecast

The Future Sales Forecast section predicts the **next 3 months of sales** using the trained Monthly Sales Forecast Regression model.

A **recursive forecasting approach** is used. This means that the predicted sales for one month are added to the historical data and then used as a lag value when predicting the following month.

### 10.1 Define Forecast Period

The number of future months to forecast is set to 3.

```python
future_steps = 3
```

This means that the model will generate sales predictions for the next **three months**.

### 10.2 Prepare Historical Sales Data

The historical monthly Sales data is copied into a separate DataFrame.

```python
history = monthly[["Order Date", "Monthly_Sales"]].copy()
```

The `history` DataFrame is used to maintain both historical and newly predicted Sales values during recursive forecasting.

### 10.3 Generate Future Months

A loop is used to forecast each future month.

```python
for _ in range(future_steps):
    next_date = history["Order Date"].max() + pd.offsets.MonthBegin(1)
```

The next forecast month is calculated by adding one month to the most recent available month.

### 10.4 Create Lag Features for Forecasting

The latest Sales values are extracted from the historical data.

```python
values = history["Monthly_Sales"].tolist()

lag1 = values[-1]
lag2 = values[-2]
lag3 = values[-3]
rolling3 = np.mean(values[-3:])
```

The forecasting features are:

* **Lag_1** → Sales from the previous month.
* **Lag_2** → Sales from two months earlier.
* **Lag_3** → Sales from three months earlier.
* **Rolling_3** → Average Sales of the previous three months.

These features are required by the trained forecasting model.

### 10.5 Create Time-Based Features

Time-related information is created for each future month.

```python
"Time_Index": len(history),
"Year": next_date.year,
"Month": next_date.month,
"Quarter": next_date.quarter,
```

These features provide the model with information about:

* The position of the future month in the time series.
* The year.
* The month.
* The quarter.

### 10.6 Predict Future Sales

The trained `forecast_model` is used to predict Sales for the next month.

```python
predicted_sales = forecast_model.predict(
    row[forecast_features]
)[0]
```

The prediction is then restricted to a minimum value of zero:

```python
predicted_sales = max(0, predicted_sales)
```

This prevents the model from producing a negative Sales forecast.

### 10.7 Recursive Forecasting

The predicted Sales value is added back into the historical dataset.

```python
history = pd.concat(
    [history, row[["Order Date", "Monthly_Sales"]]],
    ignore_index=True
)
```

This is an important part of the recursive forecasting process.

For example:

1. The model predicts the first future month.
2. That prediction becomes part of the available history.
3. The prediction is then used as `Lag_1` when forecasting the next month.
4. The process continues until all 3 future months have been predicted.

This allows the model to forecast multiple future periods even though actual future Sales values are not available.

### 10.8 Create Future Forecast Dataset

The forecasted months and Sales values are stored in a DataFrame.

```python
future_forecast = pd.DataFrame(
    future_rows,
    columns
```


In [ ]:
# ============================================================
# 11. PRODUCT-LEVEL SALES SUMMARY
# ============================================================

product_sales = (
    df.groupby(["Product ID", "Product Name"])["Sales"]
      .sum()
      .sort_values(ascending=False)
      .reset_index()
)

print("\nTop 10 products:")
print(product_sales.head(10).to_string(index=False))

## 11. Product-Level Sales Summary

Product-level analysis is performed to identify the products that generate the highest total Sales.

This analysis helps understand which individual products contribute the most to overall revenue and can support inventory and product planning decisions.

### 11.1 Calculate Total Sales by Product

The dataset is grouped by `Product ID` and `Product Name`, and the total Sales for each product is calculated.

```python
product_sales = (
    df.groupby(["Product ID", "Product Name"])["Sales"]
      .sum()
      .sort_values(ascending=False)
      .reset_index()
)
```

The analysis performs the following steps:

1. Groups the data by **Product ID** and **Product Name**.
2. Calculates the total `Sales` for each product.
3. Sorts the products in descending order of Sales.
4. Resets the index to create a structured DataFrame.

The resulting DataFrame contains:

* `Product ID` → Unique identifier of the product.
* `Product Name` → Name of the product.
* `Sales` → Total Sales generated by the product.

### 11.2 Display Top 10 Products

The top 10 products with the highest total Sales are displayed.

```python
print("\nTop 10 products:")
print(product_sales.head(10).to_string(index=False))
```

The `head(10)` function selects the first 10 products after sorting.

Therefore, the output represents the **10 highest-selling products based on total Sales**.

### 11.3 Business Purpose

Product-level Sales analysis can help businesses:

* Identify high-performing products.
* Prioritize inventory for popular products.
* Understand which products contribute significantly to revenue.
* Support purchasing and replenishment decisions.
* Identify products that may require additional stock.
* Improve product-level sales planning.

### 11.4 Conclusion

The Product-Level Sales Summary ranks products according to their total Sales.

By identifying the **top 10 products**, the business can focus on products with strong sales performance and use this information to support inventory management and future sales planning.


In [ ]:
# ============================================================
# 12. BUSINESS INSIGHTS
# ============================================================

best_category = category_sales.idxmax()
best_region = df.groupby("Region")["Sales"].sum().idxmax()
best_product = product_sales.iloc[0]["Product Name"]

print("\nBUSINESS INSIGHTS")
print("-----------------")
print("Best-selling category:", best_category)
print("Highest-sales region :", best_region)
print("Top product          :", best_product)
print("Total sales          :", round(df["Sales"].sum(), 2))
print("Total profit         :", round(df["Profit"].sum(), 2))

print("\nInterpretation:")
print("- Historical monthly sales are used to estimate future sales.")
print("- Lagged sales capture recent demand patterns.")
print("- Forecasts should support inventory planning, not replace business judgment.")
print("- Because this dataset has no Store column, forecasting is performed at overall monthly level.")

## 12. Business Insights

Business insights summarize the key findings obtained from the descriptive, diagnostic, and predictive analysis.

This section identifies the **best-selling category, highest-sales region, top product, total sales, and total profit**.

### 12.1 Identify Best-Selling Category

The category with the highest total Sales is identified using:

```python id="7qf1fr"
best_category = category_sales.idxmax()
```

The `idxmax()` function returns the category with the maximum Sales value.

This helps identify the product category that contributes the most to overall Sales.

### 12.2 Identify Highest-Sales Region

Total Sales are grouped by Region to determine the region with the highest Sales.

```python id="y2h3df"
best_region = df.groupby("Region")["Sales"].sum().idxmax()
```

The analysis:

1. Groups transactions by `Region`.
2. Calculates total Sales for each region.
3. Identifies the region with the highest total Sales.

This provides an overview of the strongest-performing geographical region.

### 12.3 Identify Top Product

The highest-selling product is obtained from the previously created `product_sales` DataFrame.

```python id="d9u5pa"
best_product = product_sales.iloc[0]["Product Name"]
```

Because the `product_sales` DataFrame was sorted in descending order of Sales, the first row represents the product with the highest total Sales.

### 12.4 Calculate Total Sales and Profit

The total Sales and Profit generated by the dataset are calculated.

```python id="8t6d3w"
print("Total sales          :", round(df["Sales"].sum(), 2))
print("Total profit         :", round(df["Profit"].sum(), 2))
```

* **Total Sales** → Sum of all Sales transactions.
* **Total Profit** → Sum of all Profit values.

The `round()` function is used to display the values with two decimal places.

### 12.5 Display Business Insights

The key findings are displayed in a structured format.

```python id="9p0v1k"
print("\nBUSINESS INSIGHTS")
print("-----------------")
print("Best-selling category:", best_category)
print("Highest-sales region :", best_region)
print("Top product          :", best_product)
print("Total sales          :", round(df["Sales"].sum(), 2))
print("Total profit         :", round(df["Profit"].sum(), 2))
```

The output provides a quick summary of the major business performance indicators.

### 12.6 Interpretation of Forecasting Results

The project uses historical Sales data to estimate future Sales.

The following points summarize the interpretation:

#### Historical Sales for Forecasting

Historical monthly Sales are used as the foundation for predicting future Sales.

Past Sales patterns provide useful information about expected future demand.

#### Lagged Sales

Lagged Sales features capture recent demand patterns.

The model uses previous months' Sales to help estimate the Sales of upcoming months.

#### Forecasts and Business Judgment

Sales forecasts are intended to support business decision-making.

Forecast results should be used as a planning aid and **should not completely replace business judgment**, market knowledge, or other relevant business information.

#### Overall Monthly Forecasting

The dataset does not contain a `Store` column.

Therefore, the forecasting model predicts Sales at the **overall monthly level** rather than producing separate forecasts for individual
